In [ ]:
import numpy as np 
import pandas as pd 


In [ ]:
df = pd.read_csv('IMDBDataset_small.csv')
# df = pd.read_csv('IMDBDataset_small.csv')
df.head()

Text preprocessing on the review

## LowerCasing

In [ ]:
df['review'][0].lower()

In [ ]:
df['review'] = df['review'].str.lower()


In [ ]:
df.head()

In [ ]:
df['review'][3]

## Removing html if present

In [ ]:
import re
def remove_html_tags(text):
    pattern = re.compile('<.*?>')
    return pattern.sub(r'',str(text))

df['review'] = df['review'].apply(remove_html_tags)

Removing url if present

In [ ]:
def remove_url(text):
    pattern = re.compile(r'https?://\S+|www\.\S+')
    return pattern.sub(r'', str(text))

df['review'] = df['review'].apply(remove_url)


Removing punctuation

In [ ]:
import string
string.punctuation

In [ ]:
exclude = string.punctuation

In [ ]:
df['review'][1]


In [ ]:
def remove_punc_fast(text):
    return text.translate(str.maketrans('', '', exclude))

df['review'] = df['review'].apply(remove_punc_fast)


In [ ]:
df['review'][1]

## ChatWord Conversion 

Not needed here but might be needed mainly on the messages kind of data

## MisSpelled Treatment

In [ ]:
from textblob import TextBlob

def fix_spelling(text):
    textBlb = TextBlob(text)

    return textBlb.correct().string

fix_spelling('ceertain')


In [ ]:
# df['review'] = df['review'].apply(fix_spelling)

## Removing Stop words 

In [ ]:
from nltk.corpus import stopwords

In [ ]:
def remove_stopwords(text):
    new_text = []
    
    for word in text.split():
        if word in stopwords.words('english'):
            new_text.append('')
        else:
            new_text.append(word)
    x = new_text[:]
    new_text.clear()
    return " ".join(x)

In [ ]:
# df['review'] = df['review'].apply(remove_stopwords)
stop_words = set(stopwords.words("english"))

df["review"] = (
    df["review"]
    .fillna("")
    .str.lower()
    .str.split()
    .apply(lambda words: " ".join(
        word for word in words
        if word not in stop_words
    ))
)


In [ ]:
df.sample()

## Tokenizations 

using spacy for tokenization

In [ ]:
import spacy
nlp = spacy.load('en_core_web_sm')

In [ ]:
def tokenize(text):
    docs = nlp(text)
    tokens =[]
    for token in docs:
        tokens.append(token)
    return tokens

# print(type(tokenize("Hello what's my name ?")))
print(tokenize("Hello what's my name ?"))
# def tokenize(text):
#     return [token.text for token in nlp.tokenizer(text)]

In [ ]:
df['tokens'] = df['review'].apply(tokenize)


In [ ]:
df.sample()
type(df['tokens'][0])

## Stemming 


In [ ]:
from nltk.stem.porter import PorterStemmer # SnowballStemmer for the Other languages this works well for the english
ps = PorterStemmer()


In [ ]:

def stem_words(tokens):
    # 1. Guard against rows that aren't lists (like NaN or float)
    if not isinstance(tokens, (list, tuple)):
        return []

    # 2. Stringify tokens and skip empty/null items to prevent the .lower() bug
    return [
        ps.stem(str(token))
        for token in tokens
        if token is not None and str(token).strip() != ""
    ]


# Apply the safe function
df["tokens"] = df["tokens"].apply(stem_words)

    # return " ".join([ps.stem(word) for word in text.split()])
test = [ "do ", "does ", "did", "doing" ]
stem_words(test)

In [ ]:
df['tokens'] = df['tokens'].apply(stem_words)


In [ ]:
df.sample()

## Lemmitizations 

Using WordNet to find the lemma (a word that always exit in language)

Using WordNet from nltk

In [ ]:
import nltk
from nltk.stem import WordNetLemmatizer
wordnet_lemmatizer = WordNetLemmatizer()
def lemmatize(tokens):
    
    return [wordnet_lemmatizer.lemmatize(word,pos='v') for word in tokens]

test = ['the', 'mice', 'is', 'running', 'quickly', 'and', 'sees', 'two', 'building', '.']
lemmatize(test)

In [ ]:
df['tokens'] = df['tokens'].apply(lemmatize)


In [ ]:
df.sample()

# Text Representation Implementation

In [ ]:
# Problem 2

# Find out the number of words in the entire corpus and also the total number of unique words(vocabulary) using just python

In [ ]:
corpus_count = 0
corpus = []
for i in df['tokens']:
    # print(i)
    corpus_count+=len(i)
    corpus.extend(i)



# uniques

vocabulary = set(corpus)




In [ ]:
print(corpus_count)

In [ ]:
# Problem 3

# Apply One Hot Encoding

from sklearn.preprocessing import OneHotEncoder

words = np.array([
    ["I"],
    ["love"],
    ["NLP"],
    ["Python"]
])

encoder = OneHotEncoder(sparse_output=False)

encoded = encoder.fit_transform(words)

print(encoder.categories_)
print(encoded)

In [ ]:
# Problem 4

# Apply bag words and find the vocabulary also find the times each word has occured

from sklearn.feature_extraction.text import CountVectorizer

documents = [
    "I love NLP",
    "I love Python",
    "Python is useful"
]

vectorizer = CountVectorizer()

X = vectorizer.fit_transform(documents)

print(vectorizer.get_feature_names_out())
print(X.toarray())


In [ ]:
# Problem 5

# Apply bag of bi-gram and bag of tri-gram and write down your observation about the dimensionality of the vocabulary

from sklearn.feature_extraction.text import CountVectorizer

documents = [
    "the movie is good",
    "the movie is not good"
]

# Unigram model
unigram_vectorizer = CountVectorizer(ngram_range=(1, 1))
X_uni = unigram_vectorizer.fit_transform(documents)

# Unigram + bigram model
ngram_vectorizer = CountVectorizer(ngram_range=(1, 2))
X_ngram = ngram_vectorizer.fit_transform(documents)

print("Unigrams:")
print(unigram_vectorizer.get_feature_names_out())

print("\nUnigrams and bigrams:")
print(ngram_vectorizer.get_feature_names_out())

In [ ]:
# Problem 6

# Apply tf-idf and find out the idf scores of words, also find out the vocabulary.

from sklearn.feature_extraction.text import TfidfVectorizer

documents = [
    "cat sat mat",
    "cat ate fish",
    "dog ate fish"
]

vectorizer = TfidfVectorizer()

X = vectorizer.fit_transform(documents)

print("Vocabulary:")
print(vectorizer.get_feature_names_out())

print("\nTF-IDF matrix:")
print(X.toarray())